# Average eulerian properties NWES
IN this notebook the avarge 
(i) temperature
(ii) salinity
(iii) flow speed
(iv) gradient of the flow
of the NWES is calculated for the period 01/09/2023 - 02/03/2024 as this is the period over which the simulation is run. 

In [ ]:
# import needed packages
import numpy as np
import xarray as xr 
import matplotlib.pyplot as plt
import xgcm
import cartopy.crs as ccrs 
import cartopy as cart
import cartopy.feature as cfeature
from datetime import datetime, timedelta

In [ ]:
starttime = datetime(2023,9,1, 0, 0, 0, 0)
endtime = datetime(2024,3,2,0,0,0,0)
dt_name_field = timedelta(days=1)
dt_field = timedelta(hours=1)


In [ ]:
# for fieldsets
field_directory = ('/storage/shared/oceanparcels/input_data/CopernicusMarineService/'
                    'OLD_NWSHELF_ANALYSIS_FORECAST_PHY_004_013/')


input_filename = ('CMEMS_v6r1_NWS_PHY_NRT_NL_01hav3D_'
                '{year_t:04d}{month_t:02d}{day_t:02d}_'
                '{year_t:04d}{month_t:02d}{day_t:02d}_'
                'R{year_tplus:04d}{month_tplus:02d}{day_tplus:02d}_HC01.nc')
# define function to make list of files
def create_filelist(input_directory, str ,time_start,time_end,dt_field):
    time = time_start
    files = []
    while (time<=time_end):
        time_tplus = time+dt_field
        y_t =time.year
        m_t = time.month
        d_t = time.day
        y_tp = time_tplus.year
        m_tp=time_tplus.month
        d_tp=time_tplus.day
        files.append(input_directory+str.format(year_t = y_t, month_t = m_t, day_t = d_t, year_tplus = y_tp, month_tplus = m_tp, day_tplus =d_tp))
        time += dt_field
        
    return files


#
depth_level_index=0

def preprocess(ds):
    return ds.isel(depth=depth_level_index)

In [ ]:
oceanfiles=create_filelist(field_directory, input_filename,
                                starttime, endtime, dt_name_field)

depth_level_index=0

def preprocess(ds):
    return ds.isel(depth=depth_level_index)

ds = xr.open_mfdataset(oceanfiles, combine='nested', concat_dim="time",preprocess= preprocess)#,drop_variables=['so','thetao'])

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection':ccrs.PlateCarree()})

pcm = ax.pcolormesh(ds.longitude, ds.latitude, ds.uo.isel(time=0)) #change to .min or .mean to calculate min or mean value over year 


In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection':ccrs.PlateCarree()})
ax.add_feature(cfeature.COASTLINE,edgecolor='lightgray')
# ax.add_feature(cfeature.LAND, color="lightgray")
pcm = ax.pcolormesh(ds.longitude, ds.latitude, ds.thetao.isel(time=0)) #change to .min or .mean to calculate min or mean value over year 
cbar = fig.colorbar(pcm, ax=ax, orientation="vertical", fraction=0.03)
cbar.set_label('max temperature [$^{\\circ}$]',fontsize=18)
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                    linewidth=1, color='gray', alpha=1, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 20}
gl.ylabel_style =  {'size': 20}

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection':ccrs.PlateCarree()})
ax.add_feature(cfeature.COASTLINE,edgecolor='lightgray')
ax.add_feature(cfeature.LAND, color="lightgray")
pcm = ax.pcolormesh(ds.longitude, ds.latitude, ds.so.isel(time=0)) #change to .min or .mean to calculate min or mean value over year 
cbar = fig.colorbar(pcm, ax=ax, orientation="vertical", fraction=0.03)
cbar.set_label('max temperature [$^{\\circ}$]',fontsize=18)
gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                    linewidth=1, color='gray', alpha=1, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 20}
gl.ylabel_style =  {'size': 20}

In [ ]:
time_days = np.arange(0,ds.time.size,1)/24
mean_T_days = ds.thetao.mean(dim=['longitude','latitude'])
mean_S_days = ds.so.mean(dim=['longitude','latitude'])



In [ ]:
fig, ax = plt.subplots()
ax.plot(time_days,mean_T_days)
ax2 = ax.twinx()
ax2.plot(time_days,mean_S_days)

In [ ]:
mean_T = ds.thetao.mean(dim=['longitude','latitude','time'])
mean_S = ds.so.mean(dim=['longitude','latitude','time'])

In [ ]:
print(mean_T.values)
print(mean_S.values)

In [ ]:
#mean velocity
speed = np.sqrt(ds.uo**2 + ds.vo**2)
mean_speed =speed.mean(dim=['longitude','latitude','time'])
print(mean_speed.values)
# mean speed is 0.25337228 so we can use this instead of 1 m/s